In [ ]:
# %%
# STEP 1: Import libraries and define file paths
# Description: 
# This script processes hot water demand for each building in the EPC dataset.
# For each building:
# - It reads the WATER_FUEL_CODE from the EPC dataset
# - It loads the hot_water_load.csv file from the HOT_WATER folder
# - It maps the hot water load entirely to the corresponding fuel type
# - It outputs a new file called "hot_water_by_fuel.csv" in the HOT_WATER folder
# The output file contains time series hot water demand split by fuel type.

import os
import pandas as pd

# File paths
epc_file = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
models_base_path = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

# Fuel types (must match EPC coding exactly)
fuel_types = [
    "electricity",
    "lpg",
    "mineral_wood",
    "mains_gas",
    "oil",
    "wood_logs",
    "wood_pellets",
    "wood_chips",
    "coal"
]

In [2]:
# %%
# STEP 2: Load EPC dataset
# Description: Read EPC data and validate required columns

epc_df = pd.read_csv(epc_file)

required_columns = ["BUILDING_REFERENCE_NUMBER", "WATER_FUEL_CODE"]

for col in required_columns:
    if col not in epc_df.columns:
        raise ValueError(f"Missing required column: {col}")

# Ensure fuel codes are lowercase for consistency
epc_df["WATER_FUEL_CODE"] = epc_df["WATER_FUEL_CODE"].str.lower()

In [3]:
# %%
# STEP 3: Process hot_water_load.csv for each building
# Description: Loop through buildings and generate hot_water_by_fuel.csv

for _, row in epc_df.iterrows():
    building_id = str(row["BUILDING_REFERENCE_NUMBER"])
    water_fuel = row["WATER_FUEL_CODE"]

    hot_water_folder = os.path.join(models_base_path, building_id, "HOT_WATER")
    input_file = os.path.join(hot_water_folder, "hot_water_load.csv")
    output_file = os.path.join(hot_water_folder, "hot_water_by_fuel.csv")

    # Skip if input file does not exist
    if not os.path.exists(input_file):
        print(f"Skipping {building_id}: hot_water_load.csv not found")
        continue

    try:
        hw_df = pd.read_csv(input_file)
    except Exception as e:
        print(f"Error reading {building_id}: {e}")
        continue

    # Validate required columns
    required_hw_cols = ["Time", "hot_water_load"]
    for col in required_hw_cols:
        if col not in hw_df.columns:
            print(f"Skipping {building_id}: missing column {col}")
            continue

    # Create output dataframe
    output_df = pd.DataFrame()
    output_df["Time"] = hw_df["Time"]

    # Initialize all fuel columns to zero
    for fuel in fuel_types:
        output_df[fuel] = 0.0

    # Assign hot water load to correct fuel
    if water_fuel in fuel_types:
        output_df[water_fuel] += hw_df["hot_water_load"]
    else:
        print(f"Warning: Unknown WATER_FUEL_CODE '{water_fuel}' for {building_id}")

    # Save output
    try:
        output_df.to_csv(output_file, index=False)
        print(f"Processed {building_id}")
    except Exception as e:
        print(f"Error saving {building_id}: {e}")

Processed 1001829067
Processed 1001951858
Processed 1001829071
Processed 1234761001
Processed 1001991633
Processed 1001664929
Processed 1001829059
Processed 1001063639
Processed 1234761000
Processed 1236594950
Processed 1001664925
Processed 1001906271
Processed 1000238911
Processed 1001829057
Processed 1234760999
Processed 1002143357
Processed 1001951854
Processed 1001829069
Processed 1002313096
Processed 1002143351
Processed 1001870854
Processed 1001870864
Processed 1002143293
Processed 1002143352
Processed 1002143353
Processed 1234647955
Processed 1002313093
Processed 1001991629
Processed 1001991628
Processed 1000238920
Processed 1001829085
Processed 1001063646
Processed 1001829058
Processed 1234806523
Processed 1001664941
Processed 1236034494
Processed 1000336709
Processed 1234761002
Processed 1002143355
Processed 1001906269
Processed 1001870855
Processed 1001664944
Processed 1000336711
Processed 1001829079
Processed 1001870859
Processed 1001664928
Processed 1234806526
Processed 100

In [4]:
# %%
# STEP 4: Summary check (optional)
# Description: Count how many hot_water_by_fuel.csv files were created

count = 0

for _, row in epc_df.iterrows():
    building_id = str(row["BUILDING_REFERENCE_NUMBER"])
    output_file = os.path.join(models_base_path, building_id, "HOT_WATER", "hot_water_by_fuel.csv")
    
    if os.path.exists(output_file):
        count += 1

print(f"Total hot_water_by_fuel.csv files created: {count}")

Total hot_water_by_fuel.csv files created: 117
